In [ ]:
import pandas as pd
import numpy as np


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/Data_Science_Project/Los Angeles/listings_detailed.csv",
    engine="python",
    on_bad_lines="skip"
)

df.head()

In [ ]:
df.columns

In [ ]:
df.isna().sum()

In [ ]:
'''
Host Characteristics:

host_is_superhost: A primary indicator of host quality.

host_response_rate / host_acceptance_rate: Measures of professional engagement.

host_identity_verified: Trust factor.

host_listings_count: Distinguishes between individual hosts and professional management companies.

host_since: Can be used to calculate "host tenure" (years of experience).


Listing Components:

room_type : Fundamental price drivers.

accommodates, bedrooms, beds, bathrooms_text: Physical capacity of the unit.

amenities: You can count the number of amenities or look for specific high-value ones (e.g., "Wifi," "Pool").

neighbourhood_cleansed: Essential for comparing across cities.


Review & Quality Metrics:

number_of_reviews: Popularity and social proof.

instant_bookable: Ease of booking.


Target Variable:

price: This is what you are predicting or estimating.

'''

In [ ]:
cols_to_keep = [
    'host_is_superhost',
    'host_response_rate',
    'host_acceptance_rate',
    'host_identity_verified',
    'host_listings_count',
    'host_since',
    'room_type',
    'accommodates',
    'bedrooms',
    'beds',
    'bathrooms_text',
    'amenities',
    'neighbourhood_cleansed',
    'number_of_reviews',
    'instant_bookable',
    'price'
]

In [ ]:
la_df = df[cols_to_keep].copy()
la_df.head()

In [ ]:
la_df.isna().sum()

In [ ]:
#host_is_superhost: * Action: Usually, if this is null, the host is not a Superhost. You can safely fill these with "f" (false).
la_df['host_is_superhost'] = la_df['host_is_superhost'].fillna('f')

In [ ]:
'''
host_response_rate & host_acceptance_rate:
-->
second "flag" column (e.g., has_response_rate: 0 or 1) to see
if not having a rate impacts price.

why: because these rates can give severe advantage to someone with 95% rate over
someone with 85% rate.

'''

la_df['has_response_rate'] = la_df['host_response_rate'].notna().astype(int)
la_df['has_acceptance_rate'] = la_df['host_acceptance_rate'].notna().astype(int)


In [ ]:
la_df.drop(columns=['host_response_rate', 'host_acceptance_rate'], inplace=True)

In [ ]:
la_df.isna().sum()

In [ ]:
la_df.head()

In [ ]:
'''
Action: Use the accommodates value to estimate.
If accommodates is 1 or 2 and bedrooms is null, it's safe to impute a 1.
'''
import numpy as np

# Logic for Bedrooms: All Nulls become 1 (treating Studios as 1-bed units)
la_df['bedrooms'] = la_df['bedrooms'].fillna(1)

# Logic for Beds: If Null, use half of 'accommodates' (rounded up)
# Example: Accommodates 1 or 2 -> 1 bed. Accommodates 3 or 4 -> 2 beds.
la_df['beds'] = la_df['beds'].fillna((la_df['accommodates'] / 2).apply(np.ceil))


In [ ]:
la_df.isna().sum()

In [ ]:
# now just remove all rows with nulls

la_df.dropna(inplace=True)

In [ ]:
la_df.isna().sum()

In [ ]:
la_df.head()

In [ ]:
#for amenities --> just count the number of amenities and make a column named --> amenities_count

la_df['amenities_count'] = la_df['amenities'].str.count(',') + 1
la_df.drop(columns=['amenities'], inplace=True)

In [ ]:
la_df.head()

In [ ]:
la_df.head()

Now we begin fixing the data (cleaning)

In [ ]:
#converting host_is_superhost, host_identity_verified, instant_bookable
la_df['host_is_superhost'] = la_df['host_is_superhost'].map({'t': 1, '  f': 0  ,'f': 0})
la_df['instant_bookable'] = la_df['instant_bookable'].map({'t': 1, 'f': 0})

In [ ]:
#look for all uniques in host_identity verified
la_df['host_identity_verified'].unique()
#now convert
la_df['host_identity_verified'] = la_df['host_identity_verified'].map({'t': 1, 'f': 0})


In [ ]:
la_df.head()

In [ ]:
#adjust the host_since --> for actual numerical value --> in years
import pandas as pd
from datetime import datetime

la_df['host_since'] = la_df['host_since'].astype(str).str[:10]

la_df['host_since'] = pd.to_datetime(la_df['host_since'], errors='coerce')

reference_date = pd.to_datetime('2026-04-11')
la_df['host_since'] = (reference_date - la_df['host_since']).dt.days / 365.25



In [ ]:
la_df.head()

In [ ]:
#chnage the room_type to be numerical --> (One-Hot Encoding) to avoid traps with 0 and 5 being numerically different
# nyc_df['room_type'].unique()

# This creates the room_type_Private room, etc. as 1s and 0s
la_df = pd.get_dummies(la_df, columns=['room_type'], dtype=int)



In [ ]:
la_df.head()

In [ ]:
la_df.isna().sum()

In [ ]:
#fix the bathrooms text

# 1. Create 'bathrooms' column by grabbing the first number found in the text
# This handles "3.5 baths", "1 bath", "11.5 shared baths", etc.
la_df['bathrooms'] = la_df['bathrooms_text'].str.extract('(\d+\.?\d*)').astype(float)

# 2. Fix the "Half-bath" cases (since they have no number, extract makes them NaN)
# If the text says "half", we just set the number to 0.5
la_df.loc[la_df['bathrooms_text'].str.contains('half', case=False, na=False), 'bathrooms'] = 0.5

# 3. Create 'is_shared_bath' (1 if it's shared, 0 if it's private)
# This is a simple "True/False" check converted to "1/0"
la_df['is_shared_bath'] = la_df['bathrooms_text'].str.contains('shared', case=False, na=False).astype(int)

# 4. Fill any remaining blanks with 1 (the most common bathroom count)
la_df['bathrooms'] = la_df['bathrooms'].fillna(1)

In [ ]:
la_df.drop(columns=['bathrooms_text'], inplace=True)

In [ ]:
la_df.isna().sum()

In [ ]:
la_df.head()

In [ ]:
#fix the neighbourhood_cleansed
la_df['neighbourhood_cleansed'].unique()

In [ ]:
neighborhood_groups = {
    'Coastal': [
        'Malibu', 'Santa Monica', 'Venice', 'Pacific Palisades', 'Manhattan Beach',
        'Hermosa Beach', 'Marina del Rey', 'Playa del Rey', 'Avalon', 'Rancho Palos Verdes',
        'San Pedro', 'Palos Verdes Estates', 'Rolling Hills Estates', 'Rolling Hills',
        'El Segundo', 'Redondo Beach', 'Del Aire', 'Unincorporated Catalina Island'
    ],
    'Westside_Hollywood': [
        'Beverly Hills', 'Bel-Air', 'West Hollywood', 'Hollywood Hills', 'Hollywood Hills West',
        'Brentwood', 'Westwood', 'Culver City', 'Beverly Grove', 'Mar Vista', 'Sawtelle',
        'Cheviot Hills', 'Beverly Crest', 'Pico-Robertson', 'Pacific Palisades', 'Century City',
        'Rancho Park', 'Cheviot Hills', 'Windsor Square', 'Hancock Park', 'Beverlywood',
        'Bel-Air', 'Universal City', 'West Los Angeles', 'Del Rey', 'Playa Vista'
    ],
    'Central_LA': [
        'Downtown', 'Koreatown', 'Hollywood', 'Silver Lake', 'Echo Park', 'Mid-Wilshire',
        'Westlake', 'Los Feliz', 'Chinatown', 'Boyle Heights', 'East Hollywood', 'Carthay',
        'Pico-Union', 'Harvard Heights', 'Larchmont', 'Arlington Heights', 'University Park',
        'West Adams', 'Jefferson Park', 'Exposition Park', 'Adams-Normandie', 'Elysian Valley',
        'Highland Park', 'Eagle Rock', 'Atwater Village', 'Glassell Park', 'Mount Washington',
        'Cypress Park', 'Elysian Park', 'Montecito Heights', 'Lincoln Heights', 'Historic South-Central'
    ],
    'The_Valley': [
        'Sherman Oaks', 'Encino', 'Northridge', 'Woodland Hills', 'Van Nuys', 'Burbank',
        'Studio City', 'Reseda', 'Canoga Park', 'North Hollywood', 'Valley Village',
        'Valley Glen', 'Van Nuys', 'Tarzana', 'Chatsworth', 'North Hills', 'Granada Hills',
        'Porter Ranch', 'San Fernando', 'Pacoima', 'Sylmar', 'Mission Hills', 'Arleta',
        'Panorama City', 'Sun Valley', 'Sunland', 'Tujunga', 'Shadow Hills', 'Lake Balboa',
        'Winnetka', 'West Hills', 'Toluca Lake', 'Porter Ranch', 'Lake View Terrace',
        'Sepulveda Basin', 'Tujunga Canyons'
    ],
    'South_LA_Harbor': [
        'Long Beach', 'Torrance', 'San Pedro', 'Carson', 'Gardena', 'Inglewood',
        'Hawthorne', 'Compton', 'Watts', 'Lennox', 'Lawndale', 'Lomita', 'Harbor City',
        'Wilmington', 'Harbor Gateway', 'West Carson', 'Athens', 'Westmont', 'Florence',
        'Florence-Firestone', 'Green Meadows', 'South Park', 'Central-Alameda', 'Vermont Square',
        'Vermont Vista', 'Vermont Knolls', 'Vermont-Slauson', 'Hyde Park', 'Leimert Park',
        'View Park-Windsor Hills', 'Baldwin Hills/Crenshaw', 'Ladera Heights', 'Gramercy Park',
        'Manchester Square', 'Chesterfield Square', 'Harvard Park', 'Walnut Park', 'Lynwood',
        'South Gate', 'Cudahy', 'Bell', 'Bell Gardens', 'Huntington Park', 'Vernon', 'Commerce',
        'Signal Hill', 'Lakewood', 'Bellflower', 'Norwalk', 'Paramount', 'Cerritos', 'Artesia',
        'Hawaiian Gardens', 'Alondra Park', 'Willowbrook', 'Rancho Dominguez'
    ],
    'East_LA_SGV': [
        'Pasadena', 'Alhambra', 'Monterey Park', 'East Los Angeles', 'El Monte',
        'West Covina', 'Diamond Bar', 'Glendale', 'South Pasadena', 'San Marino',
        'San Gabriel', 'Temple City', 'Arcadia', 'Sierra Madre', 'Monrovia', 'Duarte',
        'Azusa', 'Glendora', 'Claremont', 'La Verne', 'San Dimas', 'Pomona', 'Walnut',
        'Rowland Heights', 'Hacienda Heights', 'La Puente', 'Valinda', 'Bassett',
        'West Puente Valley', 'Avocado Heights', 'South El Monte', 'Rosemead', 'Pico Rivera',
        'Whittier', 'Montebello', 'South San Gabriel', 'East San Gabriel', 'Mayflower Village',
        'Irwindale', 'Baldwin Park', 'Citrus', 'Charter Oak', 'South San Jose Hills',
        'South Whittier', 'East Whittier', 'North Whittier', 'La Mirada', 'Santa Fe Springs',
        'Downey', 'La Habra Heights', 'Walnut Park', 'East Pasadena', 'South Whittier',
        'Altadena', 'La Crescenta-Montrose', 'La Canada Flintridge'
    ],
    'North_County': [
        'Santa Clarita', 'Lancaster', 'Palmdale', 'Castaic', 'Acton', 'Agua Dulce',
        'Stevenson Ranch', 'Val Verde', 'Castaic Canyons', 'Hasley Canyon', 'Quartz Hill',
        'Lake Los Angeles', 'Sun Village', 'Littlerock', 'Elizabeth Lake', 'Lake Hughes',
        'Leona Valley', 'Green Valley', 'Desert View Highlands', 'Northwest Antelope Valley',
        'Northeast Antelope Valley', 'Southeast Antelope Valley', 'Northwest Palmdale',
        'Agoura Hills', 'Calabasas', 'Hidden Hills', 'Westlake Village', 'Topanga',
        'Unincorporated Santa Monica Mountains', 'Unincorporated Santa Susana Mountains',
        'Ridge Route', 'Angeles Crest', 'Lopez/Kagel Canyons', 'Vincent'
    ]
}

In [ ]:
# Create the flat mapping dictionary
flat_map = {nb: region for region, nbs in neighborhood_groups.items() for nb in nbs}

# Map the column
la_df['region'] = la_df['neighbourhood_cleansed'].map(flat_map).fillna('Other_Residential')

# Final One-Hot Encoding
la_df = pd.get_dummies(la_df, columns=['region'], drop_first=True, dtype=int)

In [ ]:
la_df.drop(columns=['neighbourhood_cleansed'], inplace=True)

In [ ]:
la_df.head()

In [ ]:
la_df.columns

B. MLE (Maximum Likelihood Estimation)

Research Question: What probability distribution best describes Airbnb prices in Los Angeles, and what are the maximum likelihood estimates of its parameters?

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np

# 1. Price Cleaning (If still a string)
if la_df['price'].dtype == 'object':
    la_df['price'] = la_df['price'].str.replace('$', '').str.replace(',', '').astype(float)

# 2. Match the NYC Order
ny_order = [
    'is_shared_bath',
    'has_response_rate',
    'number_of_reviews',
    'beds',
    'host_since',
    'bathrooms',
    'host_identity_verified',
    'bedrooms',
    'amenities_count',
    'instant_bookable',
    'host_listings_count',
    'has_acceptance_rate',
    'accommodates'
]

# --- THE DATA LINKS ---
existing_cols = [c for c in ny_order if c in la_df.columns]
X = la_df[existing_cols]  # <--- LINK TO LA FEATURES
y = la_df['price']          # <--- LINK TO LA PRICES
X = sm.add_constant(X)

# 3. MLE Model Fitting
model = sm.OLS(y, X).fit() # <--- PROCESSING THE LA DATA

# 4. Extract Results
results_df = pd.DataFrame({
    'feature': model.params.index,
    'coefficient': model.params.values,
    'std_err': model.bse.values
})

plot_df = results_df[results_df['feature'] != 'const'].copy()
plot_df['feature'] = pd.Categorical(plot_df['feature'], categories=ny_order, ordered=True)
plot_df = plot_df.sort_values('feature')

# 5. Create Visualization (PINK REMOVED)
plt.figure(figsize=(12, 8)) # Removed facecolor='#fce4ec'
ax = plt.gca()

# Colors to match the look
colors = ['#5b81ac' if c > 0 else '#ec7451' for c in plot_df['coefficient']]

plt.barh(plot_df['feature'], plot_df['coefficient'],
         xerr=plot_df['std_err'] * 1.96,
         color=colors, capsize=4, edgecolor='black', linewidth=0.5)

plt.axvline(x=0, color='grey', linestyle='--', linewidth=1)

# Titles matching NYC style
plt.title('MLE (OLS) — Significant Feature Impact on LA Airbnb Price\n(with 95% confidence intervals, p < 0.05 only)', fontsize=12)
plt.xlabel('Coefficient — $ change in price per unit increase', fontsize=10)

plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.savefig('LA_Clean_MLE_Graph.png', dpi=300)
plt.show()

# Interpretation
print("--- LA MLE IMPACT SUMMARY (NYC ORDER) ---")
for _, row in plot_df[::-1].iterrows():
    sign = "+" if row['coefficient'] > 0 else ""
    print(f"{row['feature']}: {sign}${row['coefficient']:.2f}")